# Mamba2 `mamba_chunk_scan_combined` Kernel Playground

Minimal setup to run and test `mamba_chunk_scan_combined` — the core kernel of Mamba2.

## Tensor shapes
| Param | Shape | Notes |
|-------|-------|-------|
| `x` | `(batch, seqlen, nheads, headdim)` | Input |
| `dt` | `(batch, seqlen, nheads)` | Time-step deltas |
| `A` | `(nheads,)` | State transition (negative) |
| `B` | `(batch, seqlen, ngroups, dstate)` | State input |
| `C` | `(batch, seqlen, ngroups, dstate)` | State output |
| `D` | `(nheads,)` or `(nheads, headdim)` | Skip connection |
| `z` | `(batch, seqlen, nheads, headdim)` | Optional gating |
| `dt_bias` | `(nheads,)` | Bias added to dt |
| `initial_states` | `(batch, nheads, headdim, dstate)` | Optional initial state |
| **output** | `(batch, seqlen, nheads, headdim)` | Same as x |

In [ ]:
import os, sys, types

# Set CC so Triton can compile kernels
os.environ.setdefault(
    "CC",
    os.path.expanduser("~/miniconda3/envs/cutedsl/bin/x86_64-conda-linux-gnu-gcc"),
)

# Bypass mamba_ssm/__init__.py (it imports compiled C extensions we don't need)
MAMBA_ROOT = os.path.expanduser("~/mamba")
sys.path.insert(0, MAMBA_ROOT)
pkg = types.ModuleType("mamba_ssm")
pkg.__path__ = [os.path.join(MAMBA_ROOT, "mamba_ssm")]
pkg.__package__ = "mamba_ssm"
sys.modules["mamba_ssm"] = pkg

In [ ]:
import torch
from einops import rearrange
from mamba_ssm.ops.triton.ssd_combined import mamba_chunk_scan_combined

torch.set_printoptions(precision=4, sci_mode=False)
device = "cuda"
dtype = torch.float32
print(f"torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

## Configure dimensions
These match typical Mamba2 defaults (`d_model=768`, `expand=2`, `headdim=64`, `d_state=128`).

In [ ]:
# --- Dimensions ---
batch    = 2
seqlen   = 512
nheads   = 24     # d_inner / headdim  (1536 / 64)
headdim  = 64
ngroups  = 1
dstate   = 128
chunk_size = 256

## Create inputs

In [ ]:
torch.manual_seed(42)

x  = torch.randn(batch, seqlen, nheads, headdim, device=device, dtype=dtype)
dt = torch.randn(batch, seqlen, nheads, device=device, dtype=dtype)
A  = -torch.rand(nheads, device=device, dtype=dtype)          # must be negative
B  = torch.randn(batch, seqlen, ngroups, dstate, device=device, dtype=dtype)
C  = torch.randn(batch, seqlen, ngroups, dstate, device=device, dtype=dtype)

# Optional parameters (set to None to disable)
D       = torch.randn(nheads, device=device, dtype=dtype)     # skip connection
dt_bias = torch.randn(nheads, device=device, dtype=dtype)     # bias for dt
z       = None  # gating tensor, shape (batch, seqlen, nheads, headdim) if used

print(f"x: {x.shape}")
print(f"dt: {dt.shape}")
print(f"A: {A.shape}  (values: {A[:4]})")
print(f"B: {B.shape}")
print(f"C: {C.shape}")
print(f"D: {D.shape}")
print(f"dt_bias: {dt_bias.shape}")

## Run the kernel

In [ ]:
y = mamba_chunk_scan_combined(
    x, dt, A, B, C,
    chunk_size=chunk_size,
    D=D,
    z=z,
    dt_bias=dt_bias,
    dt_softplus=True,
    # dt_limit=(0.0, float("inf")),   # default, no clamping
    # initial_states=None,
    # seq_idx=None,
    # return_final_states=False,
)

print(f"Output shape: {y.shape}")
print(f"Output dtype: {y.dtype}")
print(f"y[0,0,0,:8] = {y[0, 0, 0, :8]}")

## Return final states
Useful for recurrent inference or understanding the hidden state.

In [ ]:
y2, final_states = mamba_chunk_scan_combined(
    x, dt, A, B, C,
    chunk_size=chunk_size,
    D=D,
    dt_bias=dt_bias,
    dt_softplus=True,
    return_final_states=True,
)

print(f"y2 shape:           {y2.shape}")
print(f"final_states shape: {final_states.shape}  # (batch, nheads, headdim, dstate)")
print(f"Outputs match:      {torch.allclose(y, y2)}")

## Verify against pure-Python SSD reference
The reference `ssd()` function from the Mamba2 paper — compare outputs.

In [ ]:
import torch.nn.functional as F

def segsum(x: torch.Tensor) -> torch.Tensor:
    """Stable segment sum for computing exp(cumsum) lower-triangular matrices."""
    T = x.size(-1)
    x_cumsum = torch.cumsum(x, dim=-1)
    x_segsum = x_cumsum[..., :, None] - x_cumsum[..., None, :]
    mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=0)
    x_segsum = x_segsum.masked_fill(~mask, -torch.inf)
    return x_segsum


def ssd_ref(X, A, B, C, block_len, initial_states=None):
    """
    Pure-Python chunked SSD (from the Mamba2 paper).
    X: (b, L, h, p)      A: (b, L, h)      B: (b, L, h, n)      C: (b, L, h, n)
    Returns: Y (b, L, h, p), final_state (b, h, p, n)
    """
    assert X.shape[1] % block_len == 0
    Xc, Ac, Bc, Cc = [rearrange(t, "b (c l) ... -> b c l ...", l=block_len) for t in (X, A, B, C)]
    Ac_h = rearrange(Ac, "b c l h -> b h c l")
    A_cumsum = torch.cumsum(Ac_h, dim=-1)

    # 1) Intra-chunk (diagonal blocks)
    L_mat = torch.exp(segsum(Ac_h))
    Y_diag = torch.einsum("bclhn,bcshn,bhcls,bcshp->bclhp", Cc, Bc, L_mat, Xc)

    # 2) Per-chunk boundary states
    decay_states = torch.exp(A_cumsum[:, :, :, -1:] - A_cumsum)
    states = torch.einsum("bclhn,bhcl,bclhp->bchpn", Bc, decay_states, Xc)

    # 3) Inter-chunk scan
    if initial_states is None:
        initial_states = torch.zeros_like(states[:, :1])
    states_bound = torch.cat([initial_states, states], dim=1)
    A_end_pad = F.pad(A_cumsum[:, :, :, -1], (1, 0))
    decay_chunk = torch.exp(segsum(A_end_pad))
    new_states = torch.einsum("bhzc,bchpn->bzhpn", decay_chunk, states_bound)
    states_in = new_states[:, :-1]
    final_state = new_states[:, -1]

    # 4) Boundary -> output
    state_decay_out = torch.exp(A_cumsum)
    Y_off = torch.einsum("bclhn,bchpn,bhcl->bclhp", Cc, states_in, state_decay_out)

    Y = rearrange(Y_diag + Y_off, "b c l h p -> b (c l) h p")
    return Y, final_state

In [ ]:
# Verify against pure-Python reference with small, numerically stable values.
# NOTE: The naive reference uses exp(segsum(...)) which accumulates floating point
# error differently than the Triton kernel's numerically stable implementation.
# Small differences are expected; this checks algorithmic correctness, not exact match.

torch.manual_seed(0)
_b, _L, _h, _p, _n = 1, 256, 4, 64, 64
_chunk = 64

_x  = torch.randn(_b, _L, _h, _p, device=device, dtype=dtype) * 0.1
_A  = -0.01 * torch.rand(_h, device=device, dtype=dtype)  # very small decay
_B  = torch.randn(_b, _L, 1, _n, device=device, dtype=dtype) * 0.1
_C  = torch.randn(_b, _L, 1, _n, device=device, dtype=dtype) * 0.1
_dt = torch.ones(_b, _L, _h, device=device, dtype=dtype) * 0.5

# For the reference: A_ref = dt * A (per-step log-decay)
_A_ref = _dt * _A
_B_ref = _B.expand(_b, _L, _h, _n)
_C_ref = _C.expand(_b, _L, _h, _n)

y_ref, final_ref = ssd_ref(_x, _A_ref, _B_ref, _C_ref, block_len=_chunk)

# Triton kernel with dt_softplus=False (dt used as-is)
y_triton, final_triton = mamba_chunk_scan_combined(
    _x, _dt, _A, _B, _C,
    chunk_size=_chunk,
    D=None, z=None, dt_bias=None,
    dt_softplus=False,
    return_final_states=True,
)

# Check relative error (more meaningful than absolute for varying magnitudes)
rel_err = ((y_ref - y_triton).abs() / (y_ref.abs() + 1e-6)).mean().item()
max_diff = (y_ref - y_triton).abs().max().item()
print(f"Max  abs diff:  {max_diff:.6e}")
print(f"Mean rel error: {rel_err:.6e}")
print(f"Correlation:    {torch.corrcoef(torch.stack([y_ref.flatten(), y_triton.flatten()]))[0,1]:.6f}")
print()
print("Note: Small diffs are expected -- naive ref vs numerically stable Triton kernel.")

## Gradient check
Verify backward pass works (useful before modifying the kernel).

In [ ]:
# Make inputs require grad
x_g  = x.clone().detach().requires_grad_(True)
dt_g = dt.clone().detach().requires_grad_(True)

y_g = mamba_chunk_scan_combined(
    x_g, dt_g, A, B, C,
    chunk_size=chunk_size,
    D=D,
    dt_bias=dt_bias,
    dt_softplus=True,
)

loss = y_g.sum()
loss.backward()

print(f"x grad shape:  {x_g.grad.shape}")
print(f"dt grad shape: {dt_g.grad.shape}")
print(f"x grad norm:   {x_g.grad.norm():.4f}")
print(f"dt grad norm:  {dt_g.grad.norm():.4f}")
print("Backward pass OK!")

## Benchmark

In [ ]:
import time

def benchmark(fn, warmup=10, iters=50):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / iters
    return elapsed * 1000  # ms

ms = benchmark(lambda: mamba_chunk_scan_combined(
    x, dt, A, B, C,
    chunk_size=chunk_size,
    D=D,
    dt_bias=dt_bias,
    dt_softplus=True,
))
print(f"Forward: {ms:.3f} ms  (batch={batch}, seqlen={seqlen}, nheads={nheads}, headdim={headdim})")